# Modelo de Fama-French de 3 factores
**MA2003B — Aplicación de métodos multivariados en ciencia de datos**
Prof. Dr. Jonathan Montalvo Urquizo · Octubre 2026

Estimación por mínimos cuadrados (factorización QR) de las exposiciones de tres acciones tecnológicas (NVDA, AMZN, MSFT) y del ETF sectorial de tecnología XLK a los factores de mercado, tamaño (SMB) y valor (HML).

## 1.2 Descarga y preparación de datos
Rendimientos mensuales a partir de los precios de cierre ajustados de Yahoo Finance.

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import math
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

USE_FAMAFRENCH_3 = True
tickers = ["NVDA", "AMZN", "XLK", "MSFT"]
fecha_inicio = "2000-01-01"
fecha_fin = "2026-10-01"

In [3]:
datos_diarios = yf.download(tickers, start=fecha_inicio, end=fecha_fin,
                            auto_adjust=True, progress=False)
precios_diarios = datos_diarios["Close"]
precios_mensuales = precios_diarios.resample("ME").last()
rendimientos = (precios_mensuales / precios_mensuales.shift(1)) - 1
rendimientos = rendimientos.dropna()
print(rendimientos.head())
print("Dimensiones rendimientos:", rendimientos.shape)

Ticker          AMZN      MSFT      NVDA       XLK
Date                                              
2000-02-29  0.066796 -0.086845  0.726798  0.105068
2000-03-31 -0.027223  0.188811  0.320071  0.083892
2000-04-30 -0.176306 -0.343529  0.054929 -0.091847
2000-05-31 -0.124575 -0.103047  0.280503 -0.103977
2000-06-30 -0.248383  0.278722  0.113913  0.099556
Dimensiones rendimientos: (320, 4)


Factores mensuales de Fama-French: lectura del archivo limpio, conversión de porcentajes a decimales y eliminación de filas vacías.

In [4]:
# Archivo descargado de la Data Library de Ken French (F-F_Research_Data_5_Factors_2x3)
# y editado para dejar solo la tabla de factores mensuales.
DATAFILE_PATH = "fama_french_factors_clean.csv"
factores = pd.read_csv(DATAFILE_PATH)

# Limpiar los nombres de columnas (sin espacios) y asignar fechas como índice
factores.columns = factores.columns.str.strip()
factores["YYYYMM"] = factores["YYYYMM"].astype(str).str.strip()
factores["Fecha"] = pd.to_datetime(factores["YYYYMM"], format="%Y%m")
factores = factores.set_index("Fecha")

# Conservar los factores disponibles
columnas_factores = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
factores = factores[columnas_factores]

# Convertir a formato numérico y porcentajes a decimales
factores = factores.apply(pd.to_numeric, errors="coerce") / 100

# Eliminar filas incompletas
factores = factores.dropna()

if USE_FAMAFRENCH_3:
    # Conservar solamente las columnas del modelo FF3
    factores = factores[["Mkt-RF", "SMB", "HML", "RF"]]

print("Primeras filas de los factores (Fama-French):")
print(factores.head())
print("Dimensiones factores:", factores.shape)
print("Rango de fechas:", factores.index.min(), "a", factores.index.max())

Primeras filas de los factores (Fama-French):
            Mkt-RF     SMB     HML      RF
Fecha                                     
1963-07-01 -0.0039 -0.0048 -0.0084  0.0027
1963-08-01  0.0508 -0.0080  0.0172  0.0025
1963-09-01 -0.0157 -0.0043 -0.0002  0.0027
1963-10-01  0.0254 -0.0134 -0.0003  0.0029
1963-11-01 -0.0086 -0.0085  0.0178  0.0027
Dimensiones factores: (758, 4)
Rango de fechas: 1963-07-01 00:00:00 a 2026-08-01 00:00:00


Unión de ambas fuentes usando solo los meses comunes y cálculo del rendimiento en exceso $R_i - R_f$.

In [5]:
rendimientos.index = rendimientos.index.to_period("M")
factores.index = factores.index.to_period("M")

datos = rendimientos.join(factores, how="inner").dropna()

for t in tickers:
    datos[f"{t}_exceso"] = datos[t] - datos["RF"]

print("Primeras filas combinadas:")
print(datos.head())
print("Dimensiones finales:", datos.shape)
print("Periodo comun:", datos.index.min(), "a", datos.index.max())

Primeras filas combinadas:
             AMZN      MSFT      NVDA       XLK  Mkt-RF     SMB     HML  \
Date                                                                      
2000-02  0.066796 -0.086845  0.726798  0.105068  0.0245  0.1854 -0.0973   
2000-03 -0.027223  0.188811  0.320071  0.083892  0.0521 -0.1553  0.0848   
2000-04 -0.176306 -0.343529  0.054929 -0.091847 -0.0635 -0.0479  0.0645   
2000-05 -0.124575 -0.103047  0.280503 -0.103977 -0.0439 -0.0386  0.0458   
2000-06 -0.248383  0.278722  0.113913  0.099556  0.0468  0.0995 -0.0803   

             RF  NVDA_exceso  AMZN_exceso  XLK_exceso  MSFT_exceso  
Date                                                                
2000-02  0.0043     0.722498     0.062496    0.100768    -0.091145  
2000-03  0.0047     0.315371    -0.031923    0.079192     0.184111  
2000-04  0.0046     0.050329    -0.180906   -0.096447    -0.348129  
2000-05  0.0050     0.275503    -0.129575   -0.108977    -0.108047  
2000-06  0.0040     0.109913    -

## 1.3 Estimación por mínimos cuadrados (QR)
Con $X = QR$, las ecuaciones normales $(X^TX)\hat\theta = X^Ty$ se reducen a $R\hat\theta = Q^Ty$, que se resuelve por sustitución hacia atrás (sin calcular inversas). Las funciones siguientes calculan los p-values de la $t$ de Student sin usar scipy.

In [6]:
def _betacf(a, b, x, max_iter=200, eps=3e-14):
    """Fracción continua para la función beta incompleta (Lentz)."""
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 - qab * x / qap
    if abs(d) < 1e-30:
        d = 1e-30
    d = 1.0 / d
    h = d
    for m in range(1, max_iter + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30:
            d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30:
            c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30:
            d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30:
            c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < eps:
            break
    return h


def _betai(a, b, x):
    """Función beta incompleta regularizada I_x(a,b)."""
    if x <= 0.0:
        return 0.0
    if x >= 1.0:
        return 1.0
    lbeta = math.lgamma(a + b) - math.lgamma(a) - math.lgamma(b)
    bt = math.exp(lbeta + a * math.log(x) + b * math.log(1.0 - x))
    if x < (a + 1.0) / (a + b + 2.0):
        return bt * _betacf(a, b, x) / a
    return 1.0 - bt * _betacf(b, a, 1.0 - x) / b


def t_cdf(t, df):
    """CDF de la t de Student."""
    x = df / (df + t * t)
    p = 0.5 * _betai(df / 2.0, 0.5, x)
    return 1.0 - p if t > 0 else p


def t_pvalue_dos_colas(t, df):
    """p-value bilateral."""
    return 2.0 * (1.0 - t_cdf(abs(t), df))

In [7]:
def sustitucion_atras(R, b):
    """Resuelve R x = b con R triangular superior."""
    k = R.shape[0]
    x = np.zeros_like(b, dtype=float)
    for i in range(k - 1, -1, -1):
        x[i] = (b[i] - R[i, i + 1:] @ x[i + 1:]) / R[i, i]
    return x


def sustitucion_adelante(L, b):
    """Resuelve L x = b con L triangular inferior."""
    k = L.shape[0]
    x = np.zeros_like(b, dtype=float)
    for i in range(k):
        x[i] = (b[i] - L[i, :i] @ x[:i]) / L[i, i]
    return x


def ols_qr(X, y):
    """
    Estima y = X theta + eps por MCO usando factorización QR.
    Devuelve theta, residuos, R2, SE, t-stats y p-values.
    No calcula ninguna inversa: solo sustituciones triangulares.
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    n, k = X.shape

    if n == 0:
        raise ValueError("X tiene 0 filas: revisa que 'datos' no esté vacío.")
    if n < k:
        raise ValueError(f"X tiene {n} filas y {k} columnas: no hay suficientes observaciones.")

    Q, R = np.linalg.qr(X, mode="reduced")
    if np.any(np.abs(np.diag(R)) < 1e-10):
        raise ValueError("X no tiene rango completo: revisa los factores.")

    # X = QR  =>  (X^T X) theta = X^T y  se reduce a  R theta = Q^T y
    theta = sustitucion_atras(R, Q.T @ y)

    y_hat = X @ theta
    resid = y - y_hat

    ss_res = float((resid.T @ resid)[0, 0])
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    gl = n - k
    if gl <= 0:
        raise ValueError(f"Grados de libertad no positivos: n={n}, k={k}.")
    sigma2 = ss_res / gl

    # Var(theta) = sigma2 (X^T X)^{-1} = sigma2 (R^T R)^{-1}.
    # Para la diagonal se resuelve R^T z = e_j y luego R w = z; se toma w_j.
    diag_XtX_inv = np.zeros(k)
    for j in range(k):
        e_j = np.zeros((k, 1))
        e_j[j] = 1.0
        z = sustitucion_adelante(R.T, e_j)
        w = sustitucion_atras(R, z)
        diag_XtX_inv[j] = w[j, 0]

    se = np.sqrt(sigma2 * diag_XtX_inv).reshape(-1, 1)
    t_stats = theta / se
    p_values = np.array([t_pvalue_dos_colas(float(t_stats[i, 0]), gl)
                         for i in range(k)]).reshape(-1, 1)

    return {
        "theta": theta,
        "se": se,
        "t": t_stats,
        "p": p_values,
        "resid": resid,
        "y_hat": y_hat,
        "y": y,
        "R2": r2,
        "n": n,
        "k": k,
    }

## 1.4 Estimaciones
Coeficientes $\hat\alpha, \hat\beta_M, \hat s, \hat h$ con errores estándar, estadísticos $t$, p-values y $R^2$ de cada activo.

In [8]:
# --- Verificaciones de sanidad ---
print("datos.shape:", datos.shape)
assert len(datos) > 10, f"datos tiene solo {len(datos)} filas: revisa descarga y join."
for c in ["Mkt-RF", "SMB", "HML"]:
    assert datos[c].std() > 1e-6, f"El factor {c} está constante."

# --- Matriz X común (intercepto + 3 factores) ---
X = np.column_stack([
    np.ones(len(datos)),
    datos["Mkt-RF"].values.astype(float),
    datos["SMB"].values.astype(float),
    datos["HML"].values.astype(float),
])
print("X.shape:", X.shape, "| n filas datos:", len(datos))

nombres_coef = ["alpha", "beta_M", "s_SMB", "h_HML"]

# --- Ajustar los 4 ETFs ---
resultados = {}
for t in tickers:
    y = datos[f"{t}_exceso"].values.astype(float)
    resultados[t] = ols_qr(X, y)

# --- Tabla resumen ---
filas_tabla = []
for t in tickers:
    r = resultados[t]
    fila = {"ETF": t, "R2": round(r["R2"], 4), "n": r["n"]}
    for j, nombre in enumerate(nombres_coef):
        fila[nombre]             = round(float(r["theta"][j, 0]), 4)
        fila[f"SE_{nombre}"]     = round(float(r["se"][j, 0]), 4)
        fila[f"t_{nombre}"]      = round(float(r["t"][j, 0]), 3)
        fila[f"p_{nombre}"]      = round(float(r["p"][j, 0]), 4)
    filas_tabla.append(fila)

tabla = pd.DataFrame(filas_tabla).set_index("ETF")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print(tabla)

datos.shape: (319, 12)
X.shape: (319, 4) | n filas datos: 319
          R2    n   alpha  SE_alpha  t_alpha  p_alpha  beta_M  SE_beta_M  t_beta_M  p_beta_M   s_SMB  SE_s_SMB  t_s_SMB  p_s_SMB   h_HML  SE_h_HML  t_h_HML  p_h_HML
ETF                                                                                                                                                                 
NVDA  0.3365  319  0.0270    0.0078    3.447   0.0006  1.8727     0.1777    10.536       0.0  0.4671    0.2611    1.789   0.0745 -1.0674    0.2271   -4.701      0.0
AMZN  0.3851  319  0.0133    0.0056    2.351   0.0193  1.4755     0.1281    11.516       0.0 -0.1296    0.1882   -0.689   0.4916 -1.1372    0.1637   -6.948      0.0
XLK   0.8416  319  0.0011    0.0015    0.707   0.4802  1.2776     0.0342    37.328       0.0 -0.1402    0.0503   -2.789   0.0056 -0.5575    0.0437  -12.750      0.0
MSFT  0.4334  319  0.0052    0.0035    1.504   0.1336  1.1142     0.0789    14.114       0.0 -0.3874    0.1160   

In [9]:
print("Mayor sensibilidad al mercado:", tabla["beta_M"].idxmax(),
      f"(beta_M = {tabla['beta_M'].max():.3f})")
print()
print("Exposiciones a tamaño (SMB) y valor (HML):")
print(tabla[["s_SMB", "p_s_SMB", "h_HML", "p_h_HML"]])

Mayor sensibilidad al mercado: NVDA (beta_M = 1.873)

Exposiciones a tamaño (SMB) y valor (HML):
       s_SMB  p_s_SMB   h_HML  p_h_HML
ETF                                   
NVDA  0.4671   0.0745 -1.0674      0.0
AMZN -0.1296   0.4916 -1.1372      0.0
XLK  -0.1402   0.0056 -0.5575      0.0
MSFT -0.3874   0.0009 -0.5404      0.0


### Matriz de correlaciones de los rendimientos excedentes

In [10]:
excesos = datos[[f"{t}_exceso" for t in tickers]].copy()
excesos.columns = tickers
matriz_corr = excesos.corr()
print("Matriz de correlaciones de rendimientos excedentes:")
print(matriz_corr.round(4))

fig = px.imshow(matriz_corr.round(3), text_auto=True, zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r",
                title="Correlaciones de los rendimientos excedentes")
fig.update_layout(width=520, height=480)
fig.show()

Matriz de correlaciones de rendimientos excedentes:
        NVDA    AMZN     XLK    MSFT
NVDA  1.0000  0.4148  0.6438  0.4047
AMZN  0.4148  1.0000  0.6098  0.4398
XLK   0.6438  0.6098  1.0000  0.6998
MSFT  0.4047  0.4398  0.6998  1.0000


## 1.5 Evaluación del ajuste
Rendimientos excedentes observados frente a los estimados por el modelo (línea roja: ajuste perfecto).

In [11]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f"{t}  (R²={resultados[t]['R2']:.3f}, n={resultados[t]['n']})" for t in tickers])
meses = datos.index.astype(str)

for i, t in enumerate(tickers):
    r = resultados[t]
    y_obs = r["y"].ravel()
    y_hat = r["y_hat"].ravel()
    fila, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Scatter(x=y_hat, y=y_obs, mode="markers", text=meses,
                             marker=dict(size=5, opacity=0.6),
                             hovertemplate="%{text}<br>estimado=%{x:.3f}<br>observado=%{y:.3f}<extra></extra>"),
                  row=fila, col=col)
    lims = [min(y_hat.min(), y_obs.min()), max(y_hat.max(), y_obs.max())]
    fig.add_trace(go.Scatter(x=lims, y=lims, mode="lines", hoverinfo="skip",
                             line=dict(color="red", dash="dash", width=1)),
                  row=fila, col=col)
    fig.update_xaxes(title_text="Estimado", row=fila, col=col)
    fig.update_yaxes(title_text="Observado", row=fila, col=col)

fig.update_layout(title="Fama-French 3 factores: observado vs estimado",
                  height=750, showlegend=False)
fig.show()

### Diagnóstico de residuos $\hat\varepsilon_{i,t} = R^e_{i,t} - \hat R^e_{i,t}$
- **Autocorrelación:** Durbin-Watson (≈ 2 indica que no hay autocorrelación).
- **Heterocedasticidad:** Breusch-Pagan (p < 0.05 indica varianza no constante).
- **Valores extremos:** residuos estandarizados con $|z| > 3$.

In [12]:
def durbin_watson(resid):
    """DW = sum (e_t - e_{t-1})^2 / sum e_t^2."""
    e = resid.ravel()
    return float(((np.diff(e)) ** 2).sum() / (e ** 2).sum())


def breusch_pagan(resid, X):
    """
    BP auxiliar: regresa residuos^2 sobre X (sin constantes extra),
    devuelve el estadístico n*R2 y su p-value chi2 con k-1 gl.
    Implementamos CDF chi2 con la beta incompleta.
    """
    e2 = (resid.ravel() ** 2).reshape(-1, 1)
    aux = ols_qr(X, e2)                # usamos el mismo OLS
    lm = aux["n"] * aux["R2"]
    gl = X.shape[1] - 1
    # CDF chi2(k) = I_{x/2}(k/2, 1/2) ... implementacion via gamma incompleta
    # Usamos la relacion chi2(k) = Gamma(k/2, 2); aproximacion via serie:
    p = _chi2_sf(lm, gl)
    return lm, gl, p


def _gammainc_upper(s, x):
    """Gamma incompleta superior regularizada Q(s,x) (Lentz)."""
    if x < s + 1.0:
        # serie para P(s,x)
        term = 1.0 / s
        suma = term
        n = 1
        while abs(term) > 1e-14 * abs(suma) and n < 500:
            term *= x / (s + n)
            suma += term
            n += 1
        P = suma * math.exp(-x + s * math.log(x) - math.lgamma(s))
        return 1.0 - P
    else:
        # fraccion continua para Q(s,x)
        b = x + 1.0 - s
        c = 1.0 / 1e-30
        d = 1.0 / b
        h = d
        for i in range(1, 500):
            an = -i * (i - s)
            b += 2.0
            d = an * d + b
            if abs(d) < 1e-30:
                d = 1e-30
            c = b + an / c
            if abs(c) < 1e-30:
                c = 1e-30
            d = 1.0 / d
            delta = d * c
            h *= delta
            if abs(delta - 1.0) < 1e-14:
                break
        return h * math.exp(-x + s * math.log(x) - math.lgamma(s))


def _chi2_sf(x, gl):
    """P(chi2_gl > x)"""
    return _gammainc_upper(gl / 2.0, x / 2.0)


for t in tickers:
    r = resultados[t]
    dw = durbin_watson(r["resid"])
    lm, gl, p_bp = breusch_pagan(r["resid"], X)
    e = r["resid"].ravel()
    z = (e - e.mean()) / e.std(ddof=1)
    n_out = int((np.abs(z) > 3).sum())
    print(f"{t}: DW={dw:.3f} | BP LM={lm:.3f} (gl={gl}, p={p_bp:.4f}) "
          f"| outliers(|z|>3)={n_out}")

NVDA: DW=1.746 | BP LM=9.279 (gl=3, p=0.0258) | outliers(|z|>3)=4
AMZN: DW=1.941 | BP LM=1.826 (gl=3, p=0.6094) | outliers(|z|>3)=6
XLK: DW=2.193 | BP LM=0.462 (gl=3, p=0.9272) | outliers(|z|>3)=4
MSFT: DW=2.276 | BP LM=3.019 (gl=3, p=0.3886) | outliers(|z|>3)=6


In [13]:
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f"{t}  (DW={durbin_watson(resultados[t]['resid']):.2f})" for t in tickers])
fechas = datos.index.to_timestamp()

for i, t in enumerate(tickers):
    e = resultados[t]["resid"].ravel()
    z = (e - e.mean()) / e.std(ddof=1)
    extremos = np.abs(z) > 3
    fila, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(go.Scatter(x=fechas, y=e, mode="lines",
                             line=dict(color="steelblue", width=1)),
                  row=fila, col=col)
    # Valores extremos (|z| > 3) marcados en rojo
    fig.add_trace(go.Scatter(x=fechas[extremos], y=e[extremos], mode="markers",
                             marker=dict(color="red", size=7)),
                  row=fila, col=col)
    fig.add_hline(y=0, line=dict(color="red", width=1), row=fila, col=col)
    fig.update_yaxes(title_text="residuo", row=fila, col=col)

fig.update_layout(title="Residuos del modelo FF3 a lo largo del tiempo (rojo: |z| > 3)",
                  height=700, showlegend=False)
fig.show()

### Exportar la tabla para el reporte

In [14]:
tabla.to_csv("tabla_coeficientes_ff3.csv")

def tabla_a_latex(df, decimales=3):
    cols = list(df.columns)
    out = [r"\begin{tabular}{l" + "r" * len(cols) + "}", r"\hline",
           "ETF & " + " & ".join(cols) + r" \\", r"\hline"]
    for idx, fila in df.iterrows():
        vals = [f"{fila[c]:.{decimales}f}"
                if isinstance(fila[c], (int, float, np.floating)) else str(fila[c])
                for c in cols]
        out.append(f"{idx} & " + " & ".join(vals) + r" \\")
    out += [r"\hline", r"\end{tabular}"]
    return "\n".join(out)

print(tabla_a_latex(tabla[["R2", "alpha", "beta_M", "s_SMB", "h_HML"]]))

\begin{tabular}{lrrrrr}
\hline
ETF & R2 & alpha & beta_M & s_SMB & h_HML \\
\hline
NVDA & 0.337 & 0.027 & 1.873 & 0.467 & -1.067 \\
AMZN & 0.385 & 0.013 & 1.476 & -0.130 & -1.137 \\
XLK & 0.842 & 0.001 & 1.278 & -0.140 & -0.557 \\
MSFT & 0.433 & 0.005 & 1.114 & -0.387 & -0.540 \\
\hline
\end{tabular}
